# 1.3 Output Parsers with Pydantic

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langgraph==1.2.16"

## Your API key

Some cells in this notebook call a model and need your own OpenAI API key. Paste it when asked. It is stored only in this notebook session, never saved to the file.

In [ ]:
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

## Learning objectives

- Compare the two routes to structured output: prompt-based parsing (`PydanticOutputParser`) vs provider-native structured output (`with_structured_output`).
- Define Pydantic v2 schemas whose field descriptions produce reliable format instructions.
- Wire `get_format_instructions()` into a prompt and run `prompt | llm | parser` end to end.
- Diagnose the three failure modes (bad JSON, wrong shape, silent drift) and fix them with validation and a manual retry loop.
- Choose `with_structured_output`'s method (`json_schema` default, `json_mode`, `function_calling`) and know the strict-mode constraints.

## The idea in plain words

Agents cannot act on prose. A downstream step needs a ticket's `category` as a value it can switch on, not a sentence that mentions the category. There are two ways to get the model to produce machine-shaped data:

1. **Ask politely in the prompt, parse on your side.** `PydanticOutputParser` generates format instructions describing your schema, you put them in the prompt, the model replies with JSON, and the parser validates it into a Pydantic object. Works with any provider, including local models, because nothing provider-specific is involved.
2. **Constrain the generation itself.** `model.with_structured_output(Schema)` tells the provider's API to enforce the schema during decoding. The model cannot produce invalid JSON for that schema (with strict mode), so parsing failures mostly disappear.

This lesson teaches both, in that order. The parser route is portable and makes the mechanics visible; the structured-output route is what you will reach for in production on the OpenAI API. Modules 2 and 4 lean on the second route, so learn the first route's failure modes here, because they explain why the second route exists.

## StrOutputParser in one breath

The simplest parser turns a model message into a string. Lesson 1.1 already used it; the detail to keep is that it also handles streaming chunk assembly, which is why canonical chains end with it rather than concatenating raw chunks by hand.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | ChatOpenAI(model="gpt-4o-mini") | StrOutputParser()

## Pydantic v2 schemas that actually work

The schema is the contract, and its field descriptions do real work: they are copied into the format instructions the model reads. A description like `"Customer's full name"` guides the model; a bare `name: str` leaves it guessing.

In [ ]:
from pydantic import BaseModel, Field, ConfigDict
from typing import Optional, Literal

class JobPosting(BaseModel):
    title: str = Field(description="The job title, e.g. 'Senior Python Engineer'.")
    location: str = Field(description="City and country, e.g. 'Berlin, Germany'.")
    salary_eur: Optional[int] = Field(
        default=None,
        description="Annual gross salary in EUR, if stated. Null when not mentioned.",
    )
    remote: Literal["yes", "no", "hybrid"] = Field(
        description="Whether the role is remote, on-site, or hybrid."
    )
    skills: list[str] = Field(description="Technical skills mentioned in the posting.")

Three habits pay off:

- **Describe every field.** The descriptions become the instructions. Write them as if briefing a careful contractor.
- **Use `Literal` for enums and `Optional` for maybe-absent data.** A `Literal["yes", "no", "hybrid"]` constrains the model far better than a comment saying "one of...". `Optional` with a `null` convention beats the model inventing a value for missing data.
- **Decide your extra-field policy.** By default Pydantic ignores unexpected fields silently: if the model adds `"director": "Nolan"`, it is dropped without a word. Set `model_config = ConfigDict(extra="forbid")` on the class and unexpected fields raise a validation error instead. For extraction pipelines, forbid is usually what you want; silent drops are how wrong data enters your database unnoticed.

## Wiring format instructions into the prompt

`parser.get_format_instructions()` returns a block of text describing the JSON schema. Put it in the prompt as a partial variable (lesson 1.2), so the caller only supplies the real inputs:

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_openai import ChatOpenAI

parser = PydanticOutputParser(pydantic_object=JobPosting)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract the job posting details from the text.\n{format_instructions}"),
    ("human", "{job_text}"),
]).partial(format_instructions=parser.get_format_instructions())

print(prompt.input_variables)  # ['job_text']

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = prompt | llm | parser

result = chain.invoke({
    "job_text": "Senior Python Engineer, Berlin. 95k EUR per year, hybrid. Needs Python, Postgres, and LangChain experience."
})
print(result)
# JobPosting(title='Senior Python Engineer', location='Berlin, Germany', salary_eur=95000, remote='hybrid', skills=['Python', 'Postgres', 'LangChain'])
print(type(result))  # <class 'JobPosting'>, a real object with attributes

The rendered instructions look like this (abbreviated): "The output should be formatted as a JSON instance that conforms to the JSON schema below... Here is the output schema: ```{"properties": {"title": {...}, ...}, "required": [...]}```". That block costs roughly 200 to 400 tokens on every call: the price of the portable route. `temperature=0` is the standard setting for extraction; you want the most likely completion, not creativity.

If you run without an OpenAI key, the same chain works against a local model; only the model line changes:

In [ ]:
from langchain_ollama import ChatOllama  # pip install langchain-ollama
llm = ChatOllama(model="llama3.1", temperature=0)

Local models follow format instructions less reliably than frontier models, which is exactly why the failure handling below matters more there.

## When parsing breaks: the failure taxonomy

Verified against the pinned packages, a `PydanticOutputParser` (which extends `JsonOutputParser`) fails in three distinct ways. All of them surface as `OutputParserException`, because the parser wraps Pydantic's `ValidationError` before raising:

1. **Not JSON at all.** The model returns prose, a refusal, or truncated text. The parser cannot decode it; `OutputParserException` with the raw text attached (`exc.llm_output`).
2. **JSON, wrong shape.** Valid JSON that misses a required field or uses the wrong type in a way Pydantic cannot coerce. `OutputParserException` whose message includes Pydantic's validation errors.
3. **Silent drift.** The nastiest one: the model returns valid JSON that passes validation but is wrong. Extra fields dropped (unless `extra="forbid"`), `"95000"` coerced to `95000` without complaint, a plausible-but-invented salary when the text said nothing. No exception, wrong data.

What the parser forgives, also verified: markdown code fences around the JSON (```` ```json ... ``` ````) are stripped before parsing, and surrounding whitespace is tolerated. So do not pre-strip fences yourself; the parser already handles them.

Handle failures with a try/except ladder around the chain, catching the specific exception first:

In [ ]:
from langchain_core.exceptions import OutputParserException

try:
    result = chain.invoke({"job_text": text})
except OutputParserException as exc:
    # The model produced something unparseable. Log the raw output; it is
    # the only evidence of what went wrong.
    print("Parser failed. Raw model output was:", exc.llm_output)
    raise

Never swallow this exception and continue with a default object. A default `JobPosting` with invented fields is silent drift with extra steps.

## The manual retry loop

For recoverable failures (modes 1 and 2), the standard fix is to show the model its own bad output plus the original instructions and ask it to correct itself. The retry parsers from old tutorials (`RetryOutputParser`, `RetryWithErrorOutputParser`) no longer ship with the pinned stack: they moved to the `langchain-classic` maintenance package, and new code should not depend on it. The modern form is a small explicit loop, which has the advantage that you can see exactly what it does:

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.exceptions import OutputParserException

fix_prompt = ChatPromptTemplate.from_template(
    "The completion below did not satisfy the constraints in the original prompt.\n"
    "Return ONLY the corrected JSON object, with no commentary and no code fences.\n\n"
    "Original prompt:\n{prompt}\n\n"
    "Bad completion:\n{completion}"
)

def extract_job(text: str, max_attempts: int = 3) -> JobPosting:
    prompt_value = prompt.format_prompt(job_text=text)  # keep the original instructions
    completion = None
    for attempt in range(max_attempts):
        if attempt == 0:
            completion = (prompt | llm | StrOutputParser()).invoke({"job_text": text})
        else:
            completion = (fix_prompt | llm | StrOutputParser()).invoke({
                "prompt": prompt_value.to_string(),
                "completion": completion,
            })
        try:
            return parser.parse(completion)
        except OutputParserException:
            if attempt == max_attempts - 1:
                raise
    raise RuntimeError("unreachable")

Why this works: the fix call re-sends the full format instructions alongside the bad output, so the model sees the schema and its mistake together. Budget it honestly: each retry is a full extra generation, so `max_attempts=3` can triple the cost of a failing call. Set the budget, log every retry (Module 6), and let the final failure raise rather than returning a guess.

## with_structured_output: the provider-native route

`ChatOpenAI.with_structured_output` wraps the model so `invoke` returns your schema directly, no parser step needed. Verified in langchain-openai 1.7.2, the `method` parameter defaults to `"json_schema"`:

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured = llm.with_structured_output(JobPosting)  # method="json_schema" is the default

result = structured.invoke(
    "Extract the job posting: Senior Python Engineer, Berlin, 95000 EUR per year, hybrid. Python, Postgres, LangChain."
)
print(type(result))  # <class 'JobPosting'>

The three methods:

- **`json_schema`** (default): uses OpenAI's Structured Outputs API, which constrains decoding so the output must match the schema. This is the one to use. With `strict=True`, OpenAI additionally guarantees schema adherence, but strict mode imposes constraints on your Pydantic class: fields may not have default values, and `Field()` may not carry extra metadata like `ge`/`le` bounds. If your schema needs defaults or constraints, either drop `strict` or keep the parser route.
- **`function_calling`**: the older tool-calling path. Still works, kept for compatibility; prefer `json_schema` for pure extraction.
- **`json_mode`**: only guarantees valid JSON, not schema-shaped JSON. You must put the format instructions in the prompt yourself (same as the parser route), so it combines the costs of both approaches with the guarantees of neither. Rarely the right choice now.

For code that must not raise on bad output, pass `include_raw=True`: the result becomes a dict with `raw` (the model message), `parsed` (your object or `None`), and `parsing_error` (the exception or `None`). Inspect `parsing_error` and decide, instead of catching blind.

In [ ]:
safe = llm.with_structured_output(JobPosting, include_raw=True)
out = safe.invoke("Extract the job posting: ...")
if out["parsing_error"] is not None:
    # out["raw"] holds the AIMessage for logging; out["parsed"] is None
    handle_gracefully(out)

## Which route, when

- **Any provider, RAG extraction, local models:** `PydanticOutputParser` with format instructions in the prompt. Portable, transparent, and the retry loop above is fully in your control. Costs a few hundred instruction tokens per call.
- **OpenAI API, production extraction:** `with_structured_output` with the default `json_schema` method. Schema enforced at decode time, no instruction tokens spent, failures raise loudly. Accept the strict-mode schema constraints or design around them.
- **Agents that call tools (Modules 2 and 4):** structured output on the model, with tool schemas defined the same Pydantic way. The parser route still appears for parsing tool *results* back into objects.

A light EU note, since this course keeps one: structured output makes log hygiene easier. When every extracted field is a known key, you can redact PII fields (names, emails) from agent logs before they are stored, which is the practical core of GDPR's data-minimisation principle. Unstructured prose logs cannot be redacted reliably. If your agent runs for EU users, prefer hosted inference in the EU region and keep raw model outputs out of long-term storage; parsed objects with a redaction pass are enough for debugging.

## What can go wrong

- **Silent drift is the real enemy.** Valid JSON that validates but is wrong: invented salaries, dropped fields, coerced types. `extra="forbid"`, tight `Literal` enums, and spot-checking outputs against inputs catch what exceptions cannot.
- **Field descriptions are load-bearing.** A schema with bare types produces vague instructions and vague output. If extraction quality is poor, rewrite the descriptions before touching the model or the temperature.
- **Retry loops multiply cost.** Each failed attempt is a full generation, and a pathological input can burn `max_attempts` generations per call. Cap attempts, add a circuit breaker for repeated failures, and log retries so cost spikes are visible.
- **Strict-mode schema constraints.** `with_structured_output(..., strict=True)` rejects Pydantic fields with defaults and `Field` metadata like bounds. Design schemas for strict mode up front (required fields, plain types) or stay on the parser route.
- **`json_mode` without instructions.** `method="json_mode"` guarantees JSON, not your schema. Without format instructions in the prompt you get well-formed JSON in the wrong shape, which is mode 2 failure with extra confidence.
- **Old tutorials import retry parsers from the wrong place.** `RetryOutputParser` and friends live in `langchain_classic.output_parsers` now, not in `langchain_core`. Do not add the legacy package for this; write the manual loop.

## Check your understanding

**1. Name the two routes to structured output and the main trade-off between them.**
Prompt-based parsing (`PydanticOutputParser` + format instructions): portable to any provider, transparent, costs a few hundred instruction tokens per call and needs client-side retry handling. Provider-native (`with_structured_output`, default `json_schema`): the API enforces the schema during decoding, no instruction tokens, failures raise loudly, but it is OpenAI-specific and strict mode constrains the schema.

**2. Why do field descriptions matter, and what do `Literal` and `Optional` buy you?**
Descriptions are copied into the format instructions the model reads, so they directly steer extraction quality. `Literal` constrains a field to an exact set of values better than prose can; `Optional` with an explicit null convention stops the model inventing values for absent data.

**3. The parser raises `OutputParserException`. What are the three underlying causes, and which one raises no exception at all?**
(1) output is not JSON, (2) JSON with the wrong shape, both raise `OutputParserException` (wrapping Pydantic's `ValidationError` where applicable). (3) silent drift: valid JSON that validates but is wrong (invented values, dropped extra fields, silent coercion), which raises nothing and is the most dangerous.

**4. Why does the manual retry loop re-send the original prompt text along with the bad completion?**
The fix call needs both the schema (from the original instructions) and the mistake (the bad completion) in context to correct itself. The original `PromptValue` is captured before the loop so the instructions are identical on every attempt.

**5. You set `strict=True` on `with_structured_output` and schema validation of your Pydantic class fails at setup. What is the likely cause?**
Strict mode forbids field defaults and extra `Field()` metadata such as bounds. Either simplify the schema (required fields, plain types) for strict mode or drop `strict` and use the parser route with `Field` constraints intact.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.